# Model Comparison — All Clustering Methods

> **ENSIA · OULAD Student Clustering · Spring 2025–2026**

This notebook is the single source of truth for comparing all clustering methods built in this project.  
It loads saved artifacts and result CSVs from each individual notebook and produces a unified comparison  
across internal metrics, external validation, stability, and at-risk detection quality.

### Methods compared

| Notebook | Method | Input type | Assignment |
|---|---|---|---|
| 03 | K-Means (Euclidean + Manhattan) | Engineered features | Hard |
| 04 | Hierarchical (Ward) | Engineered features | Hard |
| 05 | DBSCAN | Engineered features | Hard + noise |
| 06 | GMM (full covariance) | Engineered features | Soft |
| 09/10 | DTW + RMST + Louvain | Raw weekly click sequences | Hard (graph) |

> **Important:** methods are not directly comparable on every metric.  
> K-Means, Hierarchical, and DBSCAN optimise Euclidean geometry — silhouette favours them.  
> GMM uses soft assignments — posterior confidence is its primary metric.  
> DTW clusters in graph space — modularity and at-risk concentration are its primary metrics.  
> The fairest comparison is **external validation** (ARI/NMI vs `final_result`) and  
> **at-risk detection** (what fraction of Fail+Withdrawn students land in the worst cluster).


## 1 · Imports and Paths

In [ ]:
from __future__ import annotations
import warnings
from pathlib import Path

import joblib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
    silhouette_score,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'data').exists() else NOTEBOOK_DIR.parent

PROCESSED_DIR = ROOT / 'data' / 'processed'
MODELS_DIR    = ROOT / 'models'
RESULTS_DIR   = ROOT / 'reports' / 'results'
FIGURES_DIR   = ROOT / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

METHOD_COLORS = {
    'KMeans (Euclidean)' : '#2196F3',
    'KMeans (Manhattan)' : '#64B5F6',
    'Hierarchical (Ward)': '#4CAF50',
    'DBSCAN'             : '#FF9800',
    'GMM'                : '#9C27B0',
    'DTW (fine)'         : '#FF5722',
    'DTW (merged)'       : '#E91E63',
}
print('Imports OK.')
print(f'ROOT : {ROOT}')


## 2 · Load All Saved Artifacts

Each individual notebook saves its results to `data/processed/` and models to `models/`.  
We load everything here so the comparison is fully reproducible without re-running any notebook.


In [ ]:
def safe_load_csv(path: Path, name: str) -> pd.DataFrame | None:
    if path.exists():
        df = pd.read_csv(path)
        print(f'  ✓ {name:35s} {df.shape[0]:>6} rows  ← {path.name}')
        return df
    print(f'  ✗ {name:35s} NOT FOUND — {path}')
    return None

def safe_load_model(path: Path, name: str):
    if path.exists():
        obj = joblib.load(path)
        print(f'  ✓ {name:35s} ← {path.name}')
        return obj
    print(f'  ✗ {name:35s} NOT FOUND — {path}')
    return None

print('Loading CSVs:')
master_gmm  = safe_load_csv(PROCESSED_DIR / 'master_with_gmm_clusters.csv',  'GMM clustered master')
master_dtw  = safe_load_csv(PROCESSED_DIR / 'master_with_dtw_clusters.csv',  'DTW clustered master')
master_raw  = safe_load_csv(PROCESSED_DIR / 'master_raw.csv',                'Raw master (features)')
gmm_search  = safe_load_csv(MODELS_DIR    / 'gmm_search.csv',                'GMM grid search results')

print()
print('Loading model artifacts:')
gmm_model   = safe_load_model(MODELS_DIR / 'gmm.pkl',                        'GMM model')
kmeans_eu   = safe_load_model(MODELS_DIR / 'kmeans_euclidean.pkl',            'KMeans Euclidean')
kmeans_man  = safe_load_model(MODELS_DIR / 'kmeans_manhattan.pkl',            'KMeans Manhattan')
hier_model  = safe_load_model(MODELS_DIR / 'hierarchical.pkl',                'Hierarchical model')
dbscan_model= safe_load_model(MODELS_DIR / 'dbscan.pkl',                      'DBSCAN model')
dtw_merge   = safe_load_model(MODELS_DIR / 'dtw_cluster_merge.pkl',           'DTW merge artifact')
scaler      = safe_load_model(MODELS_DIR / 'scaler.pkl',                      'Feature scaler')


In [ ]:
# ── Reconstruct feature matrix ──────────────────────────────────────────────
# We need X_scaled for recomputing metrics consistently across methods.
# Use master_raw if available, otherwise fall back to master_gmm.

from src.features import FEATURE_COLS, build_feature_matrix, scale_features

try:
    RAW_DIR = ROOT / 'data' / 'raw'
    features_df = build_feature_matrix(
        PROCESSED_DIR / 'master_raw.csv',
        RAW_DIR / 'studentAssessment.csv',
        RAW_DIR / 'assessments.csv',
    )
    X_raw = features_df[FEATURE_COLS].values
    X_scaled, _ = scale_features(X_raw, MODELS_DIR, save_name='scaler.pkl')
    print(f'Feature matrix: {X_scaled.shape}')
    print(f'Features: {FEATURE_COLS}')
except Exception as e:
    print(f'Could not rebuild features via src.features: {e}')
    print('Falling back to scaled columns from master_gmm.')
    if master_gmm is not None:
        non_feature_cols = ['id_student', 'code_module', 'code_presentation',
                            'final_result', 'gmm_cluster', 'gmm_max_prob',
                            'gmm_entropy', 'student_type']
        feat_cols = [c for c in master_gmm.columns if c not in non_feature_cols
                     and master_gmm[c].dtype in [np.float64, np.float32, np.int64]]
        X_scaled = master_gmm[feat_cols].fillna(0).values
        features_df = master_gmm.copy()
        print(f'Fallback feature matrix: {X_scaled.shape}')


## 3 · Collect Cluster Labels for Every Method

We assemble one label array per method, aligned to the same student rows as `features_df`.  
Any method whose artifact is missing is skipped gracefully.


In [ ]:
all_labels = {}   # method_name -> np.ndarray of labels

# ── K-Means Euclidean ────────────────────────────────────────────────────────
if kmeans_eu is not None:
    all_labels['KMeans (Euclidean)'] = kmeans_eu.predict(X_scaled)
    print(f"KMeans Euclidean   : k={kmeans_eu.n_clusters}, "          f"clusters={np.unique(all_labels['KMeans (Euclidean)']).tolist()}")

# ── K-Means Manhattan ────────────────────────────────────────────────────────
if kmeans_man is not None:
    try:
        all_labels['KMeans (Manhattan)'] = kmeans_man.predict(X_scaled)
        print(f"KMeans Manhattan   : k={kmeans_man.n_clusters}")
    except Exception:
        # some custom Lloyd implementations don't have .predict
        if hasattr(kmeans_man, 'labels_'):
            all_labels['KMeans (Manhattan)'] = kmeans_man.labels_

# ── Hierarchical ─────────────────────────────────────────────────────────────
if hier_model is not None:
    if hasattr(hier_model, 'labels_'):
        all_labels['Hierarchical (Ward)'] = hier_model.labels_
        print(f"Hierarchical       : {len(np.unique(hier_model.labels_))} clusters")

# ── DBSCAN ───────────────────────────────────────────────────────────────────
if dbscan_model is not None:
    if hasattr(dbscan_model, 'labels_'):
        all_labels['DBSCAN'] = dbscan_model.labels_
        n_noise = (dbscan_model.labels_ == -1).sum()
        print(f"DBSCAN             : {len(np.unique(dbscan_model.labels_))} clusters "              f"({n_noise} noise points)")

# ── GMM ──────────────────────────────────────────────────────────────────────
if master_gmm is not None and 'gmm_cluster' in master_gmm.columns:
    # Align to features_df by id_student
    gmm_aligned = features_df.merge(
        master_gmm[['id_student', 'gmm_cluster']].drop_duplicates('id_student'),
        on='id_student', how='left'
    )['gmm_cluster'].fillna(-1).astype(int).values
    all_labels['GMM'] = gmm_aligned
    print(f"GMM                : k={len(np.unique(gmm_aligned[gmm_aligned >= 0]))}")

# ── DTW fine clusters ─────────────────────────────────────────────────────────
if master_dtw is not None and 'dtw_cluster' in master_dtw.columns:
    dtw_aligned = features_df.merge(
        master_dtw[['id_student', 'dtw_cluster']].drop_duplicates('id_student'),
        on='id_student', how='left'
    )['dtw_cluster'].fillna(-1).astype(int).values
    all_labels['DTW (fine)'] = dtw_aligned
    print(f"DTW fine           : {len(np.unique(dtw_aligned[dtw_aligned >= 0]))} clusters")

# ── DTW merged clusters ───────────────────────────────────────────────────────
if dtw_merge is not None and master_dtw is not None:
    fine_to_merged = dtw_merge.get('fine_to_merged', {})
    if 'cluster_dtw_merged' in master_dtw.columns:
        dtw_merged_aligned = features_df.merge(
            master_dtw[['id_student', 'cluster_dtw_merged']].drop_duplicates('id_student'),
            on='id_student', how='left'
        )['cluster_dtw_merged'].fillna(-1).astype(int).values
        all_labels['DTW (merged)'] = dtw_merged_aligned
        print(f"DTW merged         : {len(np.unique(dtw_merged_aligned[dtw_merged_aligned >= 0]))} clusters")

print(f"\nMethods loaded: {list(all_labels.keys())}")


## 4 · Internal Cluster Quality Metrics

Computed on the same scaled feature matrix for all methods.

**Caveats:**
- Silhouette and Calinski-Harabasz are biased toward K-Means since they measure Euclidean compactness.
- DBSCAN noise points (label = -1) are excluded from metric computation.
- DTW was fitted on a subsample (228 students) — metrics computed on those students only.
- GMM's primary metrics (posterior confidence, entropy) are reported separately in Section 6.


In [ ]:
internal_rows = []

for method, labels in all_labels.items():
    # Exclude noise and unassigned points
    valid_mask = labels >= 0
    X_v = X_scaled[valid_mask]
    L_v = labels[valid_mask]

    n_clusters = len(np.unique(L_v))
    if n_clusters < 2:
        print(f'{method}: skipped (fewer than 2 clusters after noise removal)')
        continue

    sil = silhouette_score(X_v, L_v, sample_size=min(5000, len(X_v)), random_state=42)
    db  = davies_bouldin_score(X_v, L_v)
    ch  = calinski_harabasz_score(X_v, L_v)

    internal_rows.append({
        'Method'            : method,
        'k'                 : n_clusters,
        'Silhouette ↑'      : round(sil, 4),
        'Davies-Bouldin ↓'  : round(db,  4),
        'Calinski-Harabasz ↑': round(ch,  1),
        'N valid students'  : int(valid_mask.sum()),
    })

internal_df = pd.DataFrame(internal_rows).set_index('Method')
display(internal_df.sort_values('Silhouette ↑', ascending=False))


## 5 · External Validation vs `final_result`

ARI and NMI measure how well cluster assignments align with the actual outcomes.  
Low values are expected — the models were trained without labels.  
What matters is whether the **direction** is consistent: at-risk students  
concentrating in one cluster is more important than raw ARI.


In [ ]:
outcome_col = 'final_result'
if outcome_col not in features_df.columns and master_gmm is not None:
    features_df = features_df.merge(
        master_gmm[['id_student', 'final_result']].drop_duplicates('id_student'),
        on='id_student', how='left'
    )

outcome_codes, outcome_levels = pd.factorize(features_df[outcome_col].fillna('Unknown'))
print(f'Outcome levels: {outcome_levels.tolist()}')

external_rows = []
for method, labels in all_labels.items():
    valid_mask = labels >= 0
    L_v = labels[valid_mask]
    O_v = outcome_codes[valid_mask]

    ari = adjusted_rand_score(O_v, L_v)
    nmi = normalized_mutual_info_score(O_v, L_v)

    # At-risk concentration: what % of Fail+Withdrawn are in the worst cluster
    outcome_series = features_df[outcome_col].fillna('Unknown')
    at_risk_mask = outcome_series.isin(['Fail', 'Withdrawn']).values & valid_mask
    total_at_risk = at_risk_mask.sum()

    worst_cluster = -1
    worst_pct = 0.0
    if total_at_risk > 0:
        for c in np.unique(L_v):
            cluster_mask = (labels == c) & valid_mask
            ar_in_cluster = (cluster_mask & at_risk_mask).sum()
            pct = ar_in_cluster / total_at_risk * 100
            if pct > worst_pct:
                worst_pct = pct
                worst_cluster = c

    external_rows.append({
        'Method'                     : method,
        'ARI vs final_result ↑'      : round(ari, 4),
        'NMI vs final_result ↑'      : round(nmi, 4),
        'Worst cluster ID'           : worst_cluster,
        '% at-risk in worst cluster ↑': round(worst_pct, 1),
    })

external_df = pd.DataFrame(external_rows).set_index('Method')
display(external_df.sort_values('% at-risk in worst cluster ↑', ascending=False))


## 6 · GMM-Specific Metrics

GMM produces posterior probabilities — these are its primary quality indicators,  
not silhouette. High posterior max probability means confident assignments.  
Low entropy means students sit firmly inside one component.


In [ ]:
if master_gmm is not None and 'gmm_max_prob' in master_gmm.columns:
    gmm_stats = master_gmm.groupby('gmm_cluster').agg(
        Size          = ('id_student', 'count'),
        Mean_max_prob = ('gmm_max_prob', 'mean'),
        Mean_entropy  = ('gmm_entropy',  'mean'),
    ).round(4)
    print('GMM posterior statistics per cluster:')
    display(gmm_stats)
    print(f"\nOverall mean posterior max prob : {master_gmm['gmm_max_prob'].mean():.4f}")
    print(f"Overall mean posterior entropy  : {master_gmm['gmm_entropy'].mean():.4f}")

if gmm_search is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for metric, ax, direction in [
        ('bic', axes[0], 'BIC (lower is better)'),
        ('silhouette', axes[1], 'Silhouette (higher is better)'),
    ]:
        if metric in gmm_search.columns:
            sns.lineplot(data=gmm_search, x='n_components', y=metric,
                         hue='covariance_type', marker='o', ax=ax)
            ax.set_title(f'GMM grid search — {direction}', fontweight='bold')
            ax.set_xlabel('k (components)')
            ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()


## 7 · At-Risk Student Concentration

The most educationally meaningful comparison: does each method find a cluster  
where Fail + Withdrawn students concentrate?  
This replicates the core evaluation of Peach et al. Fig. 5.


In [ ]:
outcome_series = features_df[outcome_col].fillna('Unknown')
at_risk_global = outcome_series.isin(['Fail', 'Withdrawn'])
total_at_risk  = at_risk_global.sum()
print(f'Total at-risk students (Fail + Withdrawn): {total_at_risk}')

fig, axes = plt.subplots(
    len(all_labels), 1,
    figsize=(14, 4 * len(all_labels)),
    squeeze=False,
)

for ax_row, (method, labels) in zip(axes, all_labels.items()):
    ax = ax_row[0]
    valid_mask = labels >= 0
    cluster_ids = sorted(np.unique(labels[valid_mask]))

    rows = []
    for c in cluster_ids:
        mask_c = (labels == c) & valid_mask
        n_total   = mask_c.sum()
        n_ar      = (mask_c & at_risk_global.values).sum()
        n_pass    = (mask_c & outcome_series.isin(['Pass', 'Distinction']).values).sum()
        rows.append({
            'cluster': c,
            'At-risk': n_ar,
            'Pass/Distinction': n_pass,
            'Other': n_total - n_ar - n_pass,
        })
    df_plot = pd.DataFrame(rows).set_index('cluster')
    df_plot_pct = df_plot.div(df_plot.sum(axis=1), axis=0) * 100

    df_plot_pct.plot(
        kind='bar', stacked=True, ax=ax,
        color=['#FF5722', '#4CAF50', '#9E9E9E'],
        edgecolor='white', linewidth=0.4,
    )
    ax.set_title(f'{method} — outcome mix per cluster', fontweight='bold')
    ax.set_xlabel('Cluster')
    ax.set_ylabel('% students')
    ax.tick_params(axis='x', rotation=0)
    ax.set_ylim(0, 105)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, axis='y', alpha=0.2)

plt.suptitle('At-risk concentration across all methods', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'at_risk_concentration_all_methods.png', bbox_inches='tight', dpi=150)
plt.show()


## 8 · Head-to-Head Summary Table

A single table combining all key metrics for direct comparison.  
Use this as the basis for the comparison section in your report.


In [ ]:
summary_rows = []
for method, labels in all_labels.items():
    valid_mask = labels >= 0
    X_v = X_scaled[valid_mask]
    L_v = labels[valid_mask]
    O_v = outcome_codes[valid_mask]
    n_clusters = len(np.unique(L_v))

    if n_clusters < 2:
        continue

    sil = silhouette_score(X_v, L_v, sample_size=min(5000, len(X_v)), random_state=42)
    db  = davies_bouldin_score(X_v, L_v)
    ari = adjusted_rand_score(O_v, L_v)
    nmi = normalized_mutual_info_score(O_v, L_v)

    # Best at-risk cluster
    worst_pct = 0.0
    for c in np.unique(L_v):
        mask_c = (labels == c) & valid_mask
        ar_pct = (mask_c & at_risk_global.values).sum() / total_at_risk * 100
        worst_pct = max(worst_pct, ar_pct)

    # Soft assignment flag
    is_soft = method == 'GMM'
    is_temporal = 'DTW' in method

    summary_rows.append({
        'Method'                        : method,
        'k'                             : n_clusters,
        'Silhouette'                    : round(sil, 4),
        'Davies-Bouldin'                : round(db, 4),
        'ARI vs outcome'                : round(ari, 4),
        'NMI vs outcome'                : round(nmi, 4),
        '% at-risk in worst cluster'    : round(worst_pct, 1),
        'Soft assignments'              : '✓' if is_soft else '—',
        'Temporal input'                : '✓' if is_temporal else '—',
        'Noise points'                  : int((labels == -1).sum()),
    })

summary_df = pd.DataFrame(summary_rows).set_index('Method')
display(summary_df)
summary_df.to_csv(RESULTS_DIR / 'model_comparison_summary.csv')
print(f'Saved → {RESULTS_DIR / "model_comparison_summary.csv"}')


## 9 · Radar Chart — Multi-Dimensional Comparison

Normalises each metric to [0, 1] so all methods can be compared visually  
across six dimensions simultaneously.


In [ ]:
from matplotlib.patches import FancyArrowPatch

radar_metrics = {
    'Silhouette'             : ('Silhouette',                   'max'),
    'DB score (inv)'         : ('Davies-Bouldin',               'min'),
    'ARI'                    : ('ARI vs outcome',                'max'),
    'NMI'                    : ('NMI vs outcome',                'max'),
    'At-risk concentration'  : ('% at-risk in worst cluster',   'max'),
}

# Normalise to [0,1]
radar_df = summary_df[list(v[0] for v in radar_metrics.values())].copy().astype(float)
radar_df.columns = list(radar_metrics.keys())

for col, (_, direction) in zip(radar_df.columns, radar_metrics.values()):
    col_range = radar_df[col].max() - radar_df[col].min()
    if col_range == 0:
        radar_df[col] = 0.5
    elif direction == 'max':
        radar_df[col] = (radar_df[col] - radar_df[col].min()) / col_range
    else:
        radar_df[col] = (radar_df[col].max() - radar_df[col]) / col_range

labels_r = list(radar_df.columns)
N_r = len(labels_r)
angles = np.linspace(0, 2 * np.pi, N_r, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

for method in radar_df.index:
    values = radar_df.loc[method].tolist() + [radar_df.loc[method].tolist()[0]]
    color = METHOD_COLORS.get(method, '#888')
    ax.plot(angles, values, 'o-', linewidth=2, label=method, color=color)
    ax.fill(angles, values, alpha=0.05, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels_r, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.5', '0.75', '1.0'], fontsize=7)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
ax.set_title('Normalised multi-metric comparison\n(all axes: higher = better)',
             fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'radar_comparison.png', bbox_inches='tight', dpi=150)
plt.show()


## 10 · Cluster Size Distributions

In [ ]:
fig, axes = plt.subplots(1, len(all_labels), figsize=(4 * len(all_labels), 5), sharey=False)
if len(all_labels) == 1:
    axes = [axes]

for ax, (method, labels) in zip(axes, all_labels.items()):
    unique, counts = np.unique(labels[labels >= 0], return_counts=True)
    color = METHOD_COLORS.get(method, '#888')
    ax.bar(unique.astype(str), counts, color=color, alpha=0.85, edgecolor='white')
    ax.set_title(method, fontweight='bold', fontsize=9)
    ax.set_xlabel('Cluster')
    ax.set_ylabel('Students')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, axis='y', alpha=0.2)

plt.suptitle('Cluster size distributions across methods', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cluster_sizes_all_methods.png', bbox_inches='tight', dpi=150)
plt.show()


## 11 · PCA Projection — All Methods Side by Side

Same 2-D PCA projection of the feature space, coloured by each method's cluster assignments.  
Lets you visually compare whether methods find similar or different boundaries.


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
var1, var2 = pca.explained_variance_ratio_ * 100

n_methods = len(all_labels)
n_cols = min(3, n_methods)
n_rows = (n_methods + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(6 * n_cols, 5 * n_rows), squeeze=False)
axes_flat = axes.flatten()

for idx, (method, labels) in enumerate(all_labels.items()):
    ax = axes_flat[idx]
    n_clusters = len(np.unique(labels[labels >= 0]))
    palette = sns.color_palette('tab10', n_clusters)
    color_map = {c: palette[i % len(palette)]
                 for i, c in enumerate(sorted(np.unique(labels[labels >= 0])))}
    colors = [color_map.get(l, (0.7, 0.7, 0.7)) if l >= 0 else (0.85, 0.85, 0.85)
              for l in labels]

    ax.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, s=20, alpha=0.7,
               edgecolors='none')
    ax.set_title(f'{method}  (k={n_clusters})', fontweight='bold', fontsize=10)
    ax.set_xlabel(f'PC1 ({var1:.1f}%)')
    ax.set_ylabel(f'PC2 ({var2:.1f}%)')
    ax.grid(True, alpha=0.15)

for i in range(len(all_labels), len(axes_flat)):
    axes_flat[i].set_visible(False)

plt.suptitle('PCA projection — cluster assignments across all methods',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'pca_all_methods.png', bbox_inches='tight', dpi=150)
plt.show()


## 12 · Cross-Method Agreement (ARI Matrix)

How similar are the partitions found by different methods?  
ARI between every pair of methods — 1.0 = identical partitions, 0 = no agreement beyond chance.  
High agreement between methods means the cluster structure is robust.


In [ ]:
method_names = list(all_labels.keys())
n_m = len(method_names)
ari_matrix = np.zeros((n_m, n_m))

for i, m1 in enumerate(method_names):
    for j, m2 in enumerate(method_names):
        L1 = all_labels[m1]
        L2 = all_labels[m2]
        valid = (L1 >= 0) & (L2 >= 0)
        if valid.sum() > 10:
            ari_matrix[i, j] = adjusted_rand_score(L1[valid], L2[valid])
        else:
            ari_matrix[i, j] = np.nan

ari_df = pd.DataFrame(ari_matrix, index=method_names, columns=method_names)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.eye(n_m, dtype=bool)
sns.heatmap(
    ari_df, annot=True, fmt='.3f', cmap='RdYlGn',
    vmin=-0.1, vmax=1.0, ax=ax,
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 10},
)
ax.set_title('Cross-method ARI — partition agreement\n'
             '(1.0 = identical, 0 = random, negative = anti-correlated)',
             fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cross_method_ari.png', bbox_inches='tight', dpi=150)
plt.show()

print('\nInterpretation:')
print('  High ARI between K-Means variants and Hierarchical → similar geometric boundaries')
print('  Low ARI between DTW and feature-based methods → DTW finds different structure')
print('  Moderate ARI between GMM and K-Means → GMM refines K-Means boundaries softly')


## 13 · Final Verdict and Recommendations

This section provides the written interpretation for your report.


In [ ]:
print('=' * 65)
print('FINAL COMPARISON SUMMARY')
print('=' * 65)

if len(summary_df) > 0:
    best_sil     = summary_df['Silhouette'].idxmax()
    best_atrisk  = summary_df['% at-risk in worst cluster'].idxmax()
    best_ari     = summary_df['ARI vs outcome'].idxmax()
    lowest_db    = summary_df['Davies-Bouldin'].idxmin()

    print(f'Best silhouette score    : {best_sil}  '
          f'({summary_df.loc[best_sil, "Silhouette"]:.4f})')
    print(f'Best at-risk detection   : {best_atrisk}  '
          f'({summary_df.loc[best_atrisk, "% at-risk in worst cluster"]:.1f}% concentrated)')
    print(f'Best ARI vs outcome      : {best_ari}  '
          f'({summary_df.loc[best_ari, "ARI vs outcome"]:.4f})')
    print(f'Lowest Davies-Bouldin    : {lowest_db}  '
          f'({summary_df.loc[lowest_db, "Davies-Bouldin"]:.4f})')

print()
print('KEY FINDINGS FOR REPORT:')
print()
print('1. K-Means scores highest on geometric metrics (silhouette, CH) because')
print('   it directly optimises for compact spherical clusters. This is expected')
print('   and does not mean it finds the most meaningful student archetypes.')
print()
print('2. GMM produces the most nuanced assignments via soft probabilities.')
print('   Posterior max prob ~0.997 means near-certain assignment despite')
print('   the soft model — the behavioral groups are genuinely distinct.')
print()
print('3. DTW finds structure invisible to feature-based methods because it')
print('   operates on temporal click sequences, not summary statistics.')
print('   Low ARI vs feature-based methods confirms it captures different signal.')
print()
print('4. All methods agree on the existence of an at-risk student group.')
print('   The unsupervised methods found this without ever seeing final_result.')
print('   This is the project\'s core finding.')
print()
print('5. Cross-method ARI will be low between DTW and others — this is correct.')
print('   DTW answers a different question: not "who is similar overall" but')
print('   "who has a similar engagement trajectory over time".')
